# 5단계 — 데이터 증강 (이 단계는 증강 종류 하나씩만 켠다)

4단계까지 기준선(He 초기화, 2 Conv + Dropout 0.5, Adam 1e-3, 증강 없음)은 val 정확도가 에폭과 상관없이 약 38~43%에서 평평하고 train 대 val 격차가 55%p까지 벌어졌다. 병목은 에폭이 아니라 과적합이다. 증강은 학습 이미지를 무작위로 변형해서 같은 사진을 외우기 어렵게 만드는 정규화다. 한 번에 하나만 바꾸는 원칙대로 **이번 단계는 증강 종류를 하나씩만 켠다.** 조합은 이 단계가 끝난 뒤 따로 본다.

**이번 단계부터 학습 방식이 바뀐다 — 얼리스탑.** 5에폭 고정으로는 학습이 느린 설정(증강)이 덜 학습된 채 잘려서 불리하다. 그래서 기준선(none)과 증강 줄 모두에 똑같이 얼리스탑을 건다: 상한 30에폭, val loss 감시, patience 5(최저를 갱신하지 못한 에폭이 5번 쌓이면 멈춤), 멈추면 val loss가 가장 낮았던 에폭의 가중치로 되돌린다. 기준선도 이 방식으로 다시 잰다. 앞 단계(He · 학습률 · Dropout)의 결과는 5에폭 기준으로 그대로 둔다.

| | 값 |
|---|---|
| 프레임워크 | Keras, PyTorch |
| 바꾸는 것 | 증강 종류: **none(기준선)** · flip · rotation · zoom (하나씩만) |
| 학습 방식 | 얼리스탑: 상한 30에폭 · val loss 감시 · patience 5 · 최저 에폭 가중치로 복원. 모든 줄에 똑같이 |
| 시드 | 0 한 번씩 훑어본다. 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾼다 |
| 실행 횟수 | 시드 하나에 4 × 2 = **8번** |
| 그 밖의 조건 | 4단계 기준선과 같다: He 초기화, 2 Conv(32·64) + Dropout 0.5 + Dense(11), Adam lr 1e-3, batch 32, 학습률 고정 |
| 소요 시간 | 대략 75분 (Keras ≈ 27분, PyTorch ≈ 49분, 어림값). 증강 줄은 몇 에폭에서 멈추느냐에 따라 달라진다 |

**Keras·PyTorch 셀은 4단계 노트북의 셀에서 아래 부분만 바뀌었다.** ① 증강 층·변환을 줄(arm)마다 붙인다 ② 에폭 반복을 얼리스탑으로 바꾼다 (두 프레임워크가 같은 `Stopper` 규칙을 쓴다) ③ 멈춘 뒤 최저 에폭의 가중치로 되돌려 val · test · 증강 없는 train 정확도를 다시 잰다 ④ 결과 기록 형식. 모델 구조 · 초기화 · 옵티마이저 · 데이터 불러오기는 그대로다.

## 증강이란

**학습할 때만 이미지를 무작위로 변형해서 같은 사진이 매번 조금씩 다르게 들어가게 하는 정규화 방법**이다. 모델이 사진 하나하나를 외우기 어려워지고, 변형에 상관없이 유지되는 특징을 배우게 된다. 파라미터가 없어서 모델 크기는 그대로다 (653,003개). 검증 · 평가 때는 끈다.

| 줄 | 변형 | Keras | PyTorch |
|---|---|---|---|
| none | 증강 없음 (기준선) | | |
| flip | 좌우 반전 (확률 0.5) | `RandomFlip("horizontal")` | `RandomHorizontalFlip(0.5)` |
| rotation | ±20° 회전, 빈 곳은 검정 | `RandomRotation(20/360, fill_mode="constant")` | `RandomRotation(20, fill=0)` |
| zoom | 가운데 기준으로 최대 1.25배 확대 | `RandomZoom((-0.2, 0.0))` | `RandomAffine(0, scale=(1.0, 1.25))` |

**두 프레임워크의 증강을 맞췄다.** v1에서는 Keras가 ±36° 회전과 확대를, PyTorch가 ±10° 회전만 써서 서로 다른 증강이었다. 이번에는 종류 · 범위 · 보간(bilinear) · 빈 곳 채움(검정)을 같게 하고, 증강은 128×128로 줄인 뒤에 건다. 합성 이미지 600장으로 두 프레임워크가 만드는 분포를 쟀더니 같았다 (회전 각도 표준편차 11.4 대 11.7, 확대 배율 평균 1.115 대 1.123, 반전 비율 0.49 대 0.50). 남은 차이는 난수 발생기와 확대 배율의 표본 추출 방식(Keras는 배율의 역수를, PyTorch는 배율 자체를 균등하게 뽑는다)이고, 배율 평균이 0.01 다른 정도다. 확대만 쓰는 이유는 빈 곳 채움이 필요 없어서 채움 차이가 끼지 않기 때문이다. 셀 1-b에서 두 프레임워크가 만든 이미지를 나란히 볼 수 있다.

## 예상되는 작용

- 증강을 켜면 학습이 느려진다. val loss 바닥(최저 에폭)이 기준선(Keras 2, PyTorch 3)보다 뒤로 밀리고, 얼리스탑이 멈추는 에폭도 늘어난다
- 같은 사진을 외우기 어려워져서 val loss 최저가 낮아지고 train 대 val 격차가 줄어든다
- 변형이 클수록 효과도 비용도 크다. 회전은 모서리가 검게 비어서 평가 이미지(모서리 없음)와 달라지는 부작용이 있다

## 이 단계가 바꾸지 않는 것 (한계)

- **강도는 한 값만 본다.** 회전 ±20°, 확대 1.25배 하나씩이다. 다른 강도는 보지 않는다
- **조합은 보지 않는다.** flip + rotation + zoom 같은 조합은 이 단계가 끝난 뒤 따로 한 단계로 본다
- **지금 기준선(Dropout 0.5)에서만 본다.** 증강을 넣으면 알맞은 Dropout 값이 달라질 수 있다 (순차 튜닝의 한계, 상호작용은 못 본다)
- 후보를 고르는 데 val을 쓴다. test는 보고용이다

## 결론을 내는 규칙과 예상 (결과를 보기 전에 정해 둔다)

**시드 1개(지금 기본값)에서는 판정하지 않고 차이만 본다.** 시드 0 한 번의 값은 높게 나오는 경향이 있었고, 같은 설정을 다시 돌려도 val이 1~2%p 움직인다. 줄 사이의 변화가 이보다 한참 크지 않으면 읽지 않는다.

**시드 3개를 모으면 다음 규칙으로 판정한다** (2~4단계와 같은 기준이다).

- 증강 줄 하나하나를 기준선(none)과 같은 시드끼리 뺀다. 비교하는 값은 **얼리스탑이 돌려준 모델(val loss 최저 에폭)의 val 정확도와 val loss**다. 시드 전부에서 방향이 같고, 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 때만 "뚜렷함"이다
- 어떤 줄이 val acc와 val loss **둘 다, 두 프레임워크 모두(4칸 전부)**에서 기준선보다 뚜렷하게 낫다면 그 증강을 기준선에 넣는다. 여러 개면 val acc 평균이 가장 높은 것이다. 아니면 증강 없음을 유지한다
- 4칸 전부에서 기준선이 뚜렷하게 낫다면 "기준선이 뚜렷하게 낫다"로 적는다
- 증강 없는 train 정확도, train − val 격차, test 정확도, 최저 에폭, 멈춘 에폭은 보고용이고 판정에 쓰지 않는다

**시드 1개 훑기 뒤에는 3시드로 넘길 줄을 고른다** (3단계와 같은 절차). 4칸이 전부 기준선보다 유리한 쪽인 줄만 시드 1 · 2를 더 돌리고, 나머지는 `ARMS`에서 뺀다 (none은 비교 기준이라 남겨 둔다).

**재현 확인 (기준선 줄).** none 줄은 4단계 기준선과 같은 설정이고, 얼리스탑이 멈추기 전까지는 4단계 30에폭 곡선과 같은 길을 간다. 셀 4가 4단계 CSV(`food11_v2_epochs.csv`)에서 같은 시드의 val loss 최저 에폭과 값, 그 에폭의 val 정확도를 찾아 나란히 보여 준다. 최저 에폭이 같고 값이 재실행 흔들림(val 1~2%p) 안에서 맞아야 한다. 크게 벗어나면 얼리스탑 구현이나 환경이 달라진 것이므로 줄 비교를 읽기 전에 원인부터 확인한다.

### 예상 (가설 — 결과로 확인한다)

1. **증강 줄은 기준선보다 늦게 멈춘다.** 최저 에폭이 기준선(Keras 2, PyTorch 3)보다 뒤로 밀린다. 증강이 학습을 늦추기 때문이다.
2. **val loss 최저가 기준선보다 낮아질 것이다.** 과적합이 늦게 시작하므로 바닥이 내려간다.
3. **val 정확도는 오르되 크기는 작을 수 있다.** 3단계 Dropout 0.75처럼 +2~4%p 정도가 시드 범위(Keras 2.5~5.1%p, PyTorch 1.1~2.6%p) 안에 들 수 있어서, 3시드 없이는 못 읽을 가능성이 있다.
4. **flip이 가장 약한 변형이라 효과도 가장 작을 것이다.** rotation과 zoom 사이의 우열은 예상하지 않는다. rotation은 검은 모서리의 부작용이 있다.
5. **증강 없는 train 기준 격차가 기준선보다 줄 것이다.**
6. 두 프레임워크에서 같은 방향이 나올 것이다.

## 실행 방법

위에서부터 셀을 차례로 돌린다 (셀 0 → 1 → 1-b → 2 → 3 → 4). 셀 1-b는 증강이 두 프레임워크에서 같게 나오는지 눈으로 보는 선택 셀이라 건너뛰어도 된다 (학습 없음). 자리를 비울 때는 셀 0에서 Drive 허용을 누른 뒤 "런타임 → 모두 실행"으로 이어서 돌려도 된다. 끝나면 셀 4가 요약을 낸다.

- **GPU(T4)로 돌린다.** 이전 단계가 전부 T4였다
- 셀 1의 `[계획]`에 남은 실행 수와 시간 어림이 나온다. 이 단계의 결과는 Drive의 `food11_v2_aug.csv`에 실행 하나가 끝날 때마다 한 줄씩 쌓인다. 다시 돌리면 끝난 실행은 건너뛰고 이어서 한다 (런타임이 끊기면 그 실행만 처음부터 다시 돈다). 처음부터 다시 하려면 그 파일을 지운다
- 시간을 줄이려면 셀 1의 `ARMS`에서 줄을 뺀다 (none은 비교 기준이라 남겨 둔다)
- 셀 2(Keras)와 셀 3(PyTorch)은 서로 독립이다. 셀 4(요약)는 있는 결과만으로 만든다
- 실행 하나마다 시작 줄 → 에폭 줄 → 끝 줄이 나온다. 에폭 줄 끝에 `최저 갱신` 또는 `대기 n/5`가 붙고, 대기가 5가 되면 멈춘다. 몇 에폭에서 멈출지는 줄마다 다르다
- 시드를 더하려면 셀 1의 `SEEDS`를 `(0, 1, 2)`로 바꾸고 셀 1부터 다시 돌린다. 시드 0은 건너뛰고 1·2만 돈다
- **셀 4의 출력을 통째로 붙여 주면 된다.**

## 어떻게 읽을까

셀 4는 출력과 그래프 2개를 만든다: `[결과표]`(프레임워크마다) · `[재현 확인]` · `[기준선 대비]` · 판정 줄 · 그래프 `aug_curves.png`(에폭별 곡선)와 `aug_seeds.png`(줄별 결과).

- **결과표**의 값은 얼리스탑이 돌려준 모델(val loss 최저 에폭)의 값이다 (시드가 여러 개면 평균 [최소~최대]). train 정확도는 증강 없이 다시 잰 값이라 줄끼리 비교할 수 있다. 최저 에폭은 모델을 돌려받은 에폭이고, 상한에 닿지 않았다면 멈춘 에폭은 그보다 5 뒤다. 비고의 `상한까지 안 멈춤`은 30에폭이 끝날 때까지 val loss가 내려가는 중이었다는 뜻이다 (상한이 걸림돌일 수 있다)
- **재현 확인**은 none 줄이 4단계와 맞는지 먼저 본다
- **기준선 대비**는 시드 1개이면 차이만 나오고 판정은 "판정 안 함"이다. 시드 3개가 되면 위 규칙대로 판정 줄이 나온다
- 곡선 그래프: 줄마다 색이 하나다 (점선 = 기준선). 연한 선은 시드 하나, 굵은 선은 평균이고, 표시(동그라미 · 네모 · 세모 · 마름모)는 얼리스탑이 돌려준 에폭이다
- 줄별 그래프: 점 하나가 시드 하나이고 같은 시드는 같은 가로 위치에 있다 (왼쪽부터 시드 0 · 1 · 2). 짧은 가로선이 평균이다

## 셀 0 — 데이터 복사 (최초 1회, 런타임 끊기면 다시)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Drive에서 직접 읽으면 학습이 수십 배 느려져서 로컬로 복사한다
if not os.path.isdir("/content/food11"):
    !cp -r /content/drive/MyDrive/food11 /content/food11

## 셀 1 — 공통 설정, 증강 정의, 얼리스탑 규칙, 기록 함수, 실행 계획

In [ ]:
import csv
import gc
import json
import os
import time

import numpy as np

DATA_DIR = "/content/food11"
IMG, BATCH = 128, 32
NUM_CLASSES = 11
N_PARAMS = 653_003           # 두 프레임워크 모두 이 값이어야 같은 모델이다 (증강 층은 파라미터가 없다)

# 4단계까지의 기준선 (이 단계에서 바꾸지 않는다): He 초기화, 2 Conv(32·64) + Dropout + Dense(11),
# Adam 1e-3, batch 32, 학습률 고정. Dropout은 3단계에서 0.5 유지로 정해졌다.
OPTIMIZER, LR = "adam", 1e-3
BASE_DROPOUT = 0.5

# 이번 단계부터의 학습 방식 (얼리스탑). 기준선(none)과 증강 줄 모두에 똑같이 건다.
EPOCH_CAP = 30               # 상한
PATIENCE = 5                 # val loss가 최저를 갱신하지 못한 에폭이 5번 쌓이면 멈추고, 최저였던 에폭의 가중치로 되돌린다

# 이 단계에서 바꾸는 것: 증강 종류 하나씩. none = 증강 없음 (기준선)
ARMS = ("none", "flip", "rotation", "zoom")
ROTATION_DEG = 20            # 회전: ±20도
ZOOM = 0.2                   # 확대: 가운데 기준 1.0 ~ 1/(1-0.2) = 1.25배 (확대만이라 빈 곳 채움이 필요 없다)
SEEDS = (0,)                 # (0, 1, 2)로 바꾸면 시드 0은 건너뛰고 1·2만 추가로 돈다
MIN_PER_RUN = {              # 실행 하나에 걸리는 시간 (분, 어림값). 증강 줄은 몇 에폭에서 멈추느냐에 따라 달라진다
    "keras": {"none": 4, "flip": 7.5, "rotation": 7.5, "zoom": 7.5},
    "torch": {"none": 8, "flip": 13.5, "rotation": 13.5, "zoom": 13.5},
}
assert "none" in ARMS and all(a in MIN_PER_RUN["keras"] for a in ARMS)

# 실행 하나가 끝날 때마다 한 줄씩 쌓는다. 런타임이 끊겨도 남고,
# 다시 돌리면 끝난 실행은 건너뛴다. 처음부터 다시 하려면 이 파일을 지운다.
RESULTS = "/content/drive/MyDrive/food11_v2_aug.csv"
PREV_RESULTS = "/content/drive/MyDrive/food11_v2_epochs.csv"      # 4단계 결과 (기준선 재현 확인에 쓴다)
FIELDS = ["framework", "arm", "optimizer", "lr", "dropout", "seed", "n_params", "epoch_cap", "patience",
          "minutes", "stop_epoch", "best_epoch",
          "train_acc_curve", "train_loss_curve", "val_acc_curve", "val_loss_curve",    # 에폭별 값 (JSON 리스트, 멈춘 에폭까지)
          "val_acc", "val_loss", "test_acc", "test_loss", "train_clean_acc"]           # 돌려받은 모델(최저 에폭)의 값


def keras_aug(arm, seed):
    """Keras 증강 층. 학습할 때만 켜지고 평가에서는 입력을 그대로 내보낸다. 빈 곳은 검정, 보간은 bilinear.
    seed를 직접 넘겨서, 증강 층을 만들어도 뒤따르는 층의 초기화 난수가 none 줄과 같게 한다."""
    from keras import layers
    if arm == "flip":
        return [layers.RandomFlip("horizontal", seed=seed)]
    if arm == "rotation":
        return [layers.RandomRotation(ROTATION_DEG / 360.0, fill_mode="constant", interpolation="bilinear",
                                      fill_value=0.0, seed=seed)]
    if arm == "zoom":
        return [layers.RandomZoom((-ZOOM, 0.0), fill_mode="constant", interpolation="bilinear",
                                  fill_value=0.0, seed=seed)]
    assert arm == "none", arm
    return []


def torch_aug(arm):
    """PyTorch 증강 변환 (128x128로 줄인 PIL 이미지에 건다). Keras와 같은 범위 · 보간(bilinear) · 채움(검정 0)."""
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode
    if arm == "flip":
        return [transforms.RandomHorizontalFlip(0.5)]
    if arm == "rotation":
        return [transforms.RandomRotation(ROTATION_DEG, interpolation=InterpolationMode.BILINEAR, fill=0)]
    if arm == "zoom":
        return [transforms.RandomAffine(0, scale=(1.0, 1.0 / (1.0 - ZOOM)),
                                        interpolation=InterpolationMode.BILINEAR, fill=0)]
    assert arm == "none", arm
    return []


class Stopper:
    """얼리스탑 규칙 (두 프레임워크 공통). Keras EarlyStopping(monitor="val_loss", min_delta=0)과 같은 규칙이다.
    val loss가 지금까지의 최저보다 낮아지면 최저를 갱신하고, 아니면 기다린 횟수(wait)를 하나 올린다.
    wait가 patience에 닿으면 멈춘다. 되돌릴 가중치는 호출하는 쪽이 최저를 갱신할 때마다 담아 둔다."""

    def __init__(self, patience):
        self.patience, self.best, self.best_epoch, self.wait = patience, float("inf"), 0, 0

    def update(self, epoch, val_loss):
        """epoch는 1부터 센다. 반환: (최저를 갱신했는가, 멈춰야 하는가)"""
        if val_loss < self.best:
            self.best, self.best_epoch, self.wait = val_loss, epoch, 0
            return True, False
        self.wait += 1
        return False, self.wait >= self.patience


def epoch_line(epoch, train_acc, train_loss, val_acc, val_loss, stopper, t0):
    if stopper.wait == 0:
        status = "최저 갱신"
    else:
        status = f"대기 {stopper.wait}/{stopper.patience}" + (" → 멈춤" if stopper.wait >= stopper.patience else "")
    return (f"  에폭 {epoch}  train {train_acc:.3f} (loss {train_loss:.3f})  ·  "
            f"val {val_acc:.3f} (loss {val_loss:.3f})  ·  {status}  ·  {(time.time() - t0) / 60:.1f}분")


def usable(r):
    """같은 Dropout · 얼리스탑 설정의 결과만 쓴다 (설정을 바꾸고 다시 돌려도 예전 결과와 섞이지 않는다)."""
    return (float(r["dropout"]) == BASE_DROPOUT and int(r["epoch_cap"]) == EPOCH_CAP
            and int(r["patience"]) == PATIENCE)


def done_runs():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS, newline="", encoding="utf-8") as f:
        return {(r["framework"], r["arm"], int(r["seed"])) for r in csv.DictReader(f) if usable(r)}


def save_row(row):
    is_new = not os.path.exists(RESULTS)
    with open(RESULTS, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if is_new:
            w.writeheader()
        w.writerow(row)


def make_row(framework, arm, seed, n_params, minutes, stopper, hist, final):
    """hist: 에폭별 리스트 4개(멈춘 에폭까지), final: 되돌린 모델의 val · test · 증강 없는 train 값"""
    return dict(framework=framework, arm=arm, optimizer=OPTIMIZER, lr=LR, dropout=BASE_DROPOUT, seed=seed,
                n_params=n_params, epoch_cap=EPOCH_CAP, patience=PATIENCE, minutes=round(minutes, 2),
                stop_epoch=len(hist["val_loss"]), best_epoch=stopper.best_epoch,
                train_acc_curve=json.dumps(hist["train_acc"]), train_loss_curve=json.dumps(hist["train_loss"]),
                val_acc_curve=json.dumps(hist["val_acc"]), val_loss_curve=json.dumps(hist["val_loss"]),
                val_acc=final["val_acc"], val_loss=final["val_loss"], test_acc=final["test_acc"],
                test_loss=final["test_loss"], train_clean_acc=final["train_clean_acc"])


def tag(framework, arm, seed):
    return f"[{framework} · {arm} · 시드 {seed}]"


def report(row):
    loss = float(row["val_loss"])
    print(f"{tag(row['framework'], row['arm'], row['seed'])} 최저 에폭 {row['best_epoch']} · {row['stop_epoch']}에폭에서 멈춤 · "
          f"val {float(row['val_acc']):.3f} (loss {loss:.2f}) · test {float(row['test_acc']):.3f} · {row['minutes']:.1f}분"
          + ("" if np.isfinite(loss) else " · 발산(loss NaN)"))


def plan():
    done = done_runs()
    print(f"[계획] 증강 {', '.join(ARMS)} × 시드 {', '.join(map(str, SEEDS))} — 얼리스탑(상한 {EPOCH_CAP}에폭, patience {PATIENCE}), "
          f"Dropout {BASE_DROPOUT}, CSV에 이미 있는 실행은 건너뛴다")
    for fw, name in (("keras", "Keras"), ("torch", "PyTorch")):
        todo = [(a, s) for s in SEEDS for a in ARMS if (fw, a, s) not in done]
        minutes = sum(MIN_PER_RUN[fw][a] for a, _ in todo)
        print(f"  {name:8s} 이미 {len(SEEDS) * len(ARMS) - len(todo)}번 · 남은 {len(todo)}번 ≈ {int(minutes + 0.5)}분")


plan()

## 셀 1-b — 증강 미리보기 (선택, 학습 없음)

In [ ]:
import glob
import random

import keras
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

files = sorted(p for p in glob.glob(f"{DATA_DIR}/training/*/*") if p.lower().endswith((".jpg", ".jpeg", ".png")))
img = Image.open(random.Random(0).choice(files)).convert("RGB").resize((IMG, IMG))
arr = np.asarray(img, dtype="float32") / 255.0

shown = [a for a in ARMS if a != "none"]
assert shown, "ARMS에 증강 줄이 없다"
cols = ["original"] + [f"Keras {i}" for i in (1, 2, 3)] + [f"PyTorch {i}" for i in (1, 2, 3)]
fig, axes = plt.subplots(len(shown), len(cols), figsize=(1.7 * len(cols), 1.9 * len(shown)), squeeze=False)
for r, arm in enumerate(shown):
    k_model = keras.Sequential([keras.Input(shape=(IMG, IMG, 3)), *keras_aug(arm, 0)])
    t_tf = transforms.Compose(torch_aug(arm))
    shots = ([arr]
             + [keras.ops.convert_to_numpy(k_model(arr[None], training=True))[0] for _ in range(3)]
             + [np.asarray(t_tf(img), dtype="float32") / 255.0 for _ in range(3)])
    for c, shot in enumerate(shots):
        ax = axes[r][c]
        ax.imshow(np.clip(shot, 0, 1))
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(cols[c], fontsize=9)
        if c == 0:
            ax.set_ylabel(arm, fontsize=10)
plt.tight_layout()
plt.show()

## 셀 2 — Keras (증강 4줄, 얼리스탑) · 시드 하나에 약 27분

In [ ]:
import keras
from keras import layers


def make_optimizer(name, lr):
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=lr)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    return keras.optimizers.SGD(learning_rate=lr)


def run_keras(arm, seed):
    keras.utils.set_random_seed(seed)

    def load(split, shuffle):
        ds = keras.utils.image_dataset_from_directory(
            f"{DATA_DIR}/{split}", image_size=(IMG, IMG),
            batch_size=BATCH, shuffle=shuffle, seed=seed)
        rescale = layers.Rescaling(1.0 / 255)
        return ds.map(lambda x, y: (rescale(x), y))

    train_ds = load("training", True)
    train_eval_ds = load("training", False)       # 증강 없이 train 정확도를 다시 재는 용도
    val_ds = load("validation", False)
    test_ds = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = keras.Sequential([
        keras.Input(shape=(IMG, IMG, 3)),
        *keras_aug(arm, seed),                    # 학습할 때만 켜지는 증강 층 (none이면 없다)

        layers.Conv2D(32, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation="relu", kernel_initializer="he_normal"),
        layers.MaxPooling2D(2),

        layers.Flatten(),
        layers.Dropout(BASE_DROPOUT),
        layers.Dense(NUM_CLASSES, activation="softmax",
                     kernel_initializer="he_normal"),
    ])
    n_params = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    assert n_params == N_PARAMS, n_params

    model.compile(
        optimizer=make_optimizer(OPTIMIZER, LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    stopper, best = Stopper(PATIENCE), {}
    t0 = time.time()

    def on_epoch(epoch, logs):                    # 에폭이 끝날 때마다: 최저를 갱신하면 가중치를 담고, 멈출 때가 되면 멈춘다
        improved, stop = stopper.update(epoch + 1, logs["val_loss"])
        if improved:
            best["weights"] = model.get_weights()
        if stop:
            model.stop_training = True
        print(epoch_line(epoch + 1, logs["accuracy"], logs["loss"], logs["val_accuracy"],
                         logs["val_loss"], stopper, t0), flush=True)

    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCH_CAP, verbose=0,
                  callbacks=[keras.callbacks.LambdaCallback(on_epoch_end=on_epoch)]).history
    if "weights" in best:
        model.set_weights(best["weights"])        # 최저였던 에폭의 가중치로 되돌린다
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    _, train_clean = model.evaluate(train_eval_ds, verbose=0)
    hist = {"train_acc": h["accuracy"], "train_loss": h["loss"],
            "val_acc": h["val_accuracy"], "val_loss": h["val_loss"]}
    row = make_row("keras", arm, seed, n_params, (time.time() - t0) / 60, stopper,
                   {k: [float(v) for v in vs] for k, vs in hist.items()},
                   dict(val_loss=float(val_loss), val_acc=float(val_acc), test_loss=float(test_loss),
                        test_acc=float(test_acc), train_clean_acc=float(train_clean)))
    keras.backend.clear_session()
    gc.collect()
    return row


for seed in SEEDS:
    for arm in ARMS:
        if ("keras", arm, seed) in done_runs():
            print(f"{tag('keras', arm, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('keras', arm, seed)} 시작 (약 {MIN_PER_RUN['keras'][arm]}분)", flush=True)
        row = run_keras(arm, seed)
        save_row(row)
        report(row)

## 셀 3 — PyTorch (증강 4줄, 얼리스탑) · 시드 하나에 약 49분

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

tf_ = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),          # 0~1 변환 + HWC -> CHW
])


def make_optimizer(name, params, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.SGD(params, lr=lr)


def run_torch(arm, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    g = torch.Generator().manual_seed(seed)

    # 증강은 train에만 건다. 128x128로 줄인 뒤에 걸어서 Keras와 같은 해상도에서 변형한다 (none이면 tf_와 같다)
    train_tf = transforms.Compose([transforms.Resize((IMG, IMG)), *torch_aug(arm), transforms.ToTensor()])

    def load(split, shuffle, tf=tf_):
        ds = datasets.ImageFolder(f"{DATA_DIR}/{split}", transform=tf)
        return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                          generator=g if shuffle else None,
                          num_workers=2, pin_memory=True)

    train_dl = load("training", True, train_tf)
    train_eval_dl = load("training", False)       # 증강 없이 train 정확도를 다시 재는 용도
    val_dl = load("validation", False)
    test_dl = load("evaluation", False)

    # 128 -> conv 126 -> pool 63 -> conv 61 -> pool 30
    model = nn.Sequential(
        nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Dropout(BASE_DROPOUT),
        nn.Linear(64 * 30 * 30, NUM_CLASSES),
    ).to(device)

    for m in model:       # 1단계 — He 초기화. 가중치만, bias는 PyTorch 기본값 그대로
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert n_params == N_PARAMS, n_params

    loss_fn = nn.CrossEntropyLoss()
    optimizer = make_optimizer(OPTIMIZER, model.parameters(), LR)

    def evaluate(dataloader):
        model.eval()
        total, correct, n = 0.0, 0, 0
        with torch.no_grad():
            for X, y in dataloader:
                X, y = X.to(device), y.to(device)
                pred = model(X)
                total += loss_fn(pred, y).item() * y.size(0)
                correct += (pred.argmax(1) == y).sum().item()
                n += y.size(0)
        return total / n, correct / n

    hist = {"train_acc": [], "train_loss": [], "val_acc": [], "val_loss": []}
    stopper, best_state = Stopper(PATIENCE), None
    t0 = time.time()
    for epoch in range(EPOCH_CAP):
        model.train()
        total, correct, n = 0.0, 0, 0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total += loss.item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            n += y.size(0)
        val_loss, val_acc = evaluate(val_dl)
        hist["train_acc"].append(correct / n)
        hist["train_loss"].append(total / n)
        hist["val_acc"].append(val_acc)
        hist["val_loss"].append(val_loss)
        improved, stop = stopper.update(epoch + 1, val_loss)
        if improved:                              # 최저를 갱신하면 가중치를 담아 둔다
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(epoch_line(epoch + 1, hist["train_acc"][-1], hist["train_loss"][-1], val_acc, val_loss,
                         stopper, t0), flush=True)
        if stop:
            break

    if best_state is not None:
        model.load_state_dict(best_state)         # 최저였던 에폭의 가중치로 되돌린다
    val_loss, val_acc = evaluate(val_dl)
    test_loss, test_acc = evaluate(test_dl)
    _, train_clean = evaluate(train_eval_dl)
    row = make_row("torch", arm, seed, n_params, (time.time() - t0) / 60, stopper, hist,
                   dict(val_loss=val_loss, val_acc=val_acc, test_loss=test_loss,
                        test_acc=test_acc, train_clean_acc=train_clean))
    del model, optimizer, train_dl, train_eval_dl, val_dl, test_dl
    torch.cuda.empty_cache()
    gc.collect()
    return row


for seed in SEEDS:
    for arm in ARMS:
        if ("torch", arm, seed) in done_runs():
            print(f"{tag('torch', arm, seed)} 이미 있음 — 건너뜀")
            continue
        print(f"{tag('torch', arm, seed)} 시작 (약 {MIN_PER_RUN['torch'][arm]}분)", flush=True)
        row = run_torch(arm, seed)
        save_row(row)
        report(row)

## 셀 4 — 요약과 판정 (이 셀의 출력을 붙여 주면 된다)

In [ ]:
import unicodedata
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if usable(r) and r["arm"] in ARMS]
assert rows, "이 설정(Dropout, 얼리스탑)의 결과가 아직 없다 — 셀 2·3을 먼저 돌릴 것"
assert all(int(r["n_params"]) == N_PARAMS for r in rows), "파라미터 수가 다른 실행이 있다"

FW = {"keras": "Keras", "torch": "PyTorch"}
BASE_ARM = "none"
MIN_SEEDS = 3                  # 규칙을 적용하려면 조건마다 이만큼의 시드가 필요하다


def width(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1 for c in s)


def pad(s, w):
    return s + " " * max(0, w - width(s))


def bar(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def metrics(r):
    """얼리스탑이 돌려준 모델(최저 에폭)의 값."""
    loss = float(r["val_loss"])
    best, stop = int(r["best_epoch"]), int(r["stop_epoch"])
    val_acc, train = float(r["val_acc"]) * 100, float(r["train_clean_acc"]) * 100
    return dict(train_acc=train, val_acc=val_acc, gap=train - val_acc, val_loss=loss,
                best_ep=best, stop_ep=stop, test_acc=float(r["test_acc"]) * 100,
                capped=bool(stop >= EPOCH_CAP and EPOCH_CAP - best < PATIENCE),
                diverged=bool(best == 0 or not np.isfinite(loss)))


runs = {}      # (프레임워크, 줄) -> {시드: 지표}
curves = {}    # (프레임워크, 줄) -> {시드: 에폭별 곡선}
for r in rows:
    k, s = (r["framework"], r["arm"]), int(r["seed"])
    runs.setdefault(k, {})[s] = metrics(r)
    curves.setdefault(k, {})[s] = dict(val_loss=np.array(json.loads(r["val_loss_curve"]), dtype=float),
                                       val_acc=np.array(json.loads(r["val_acc_curve"]), dtype=float) * 100,
                                       best=int(r["best_epoch"]))

fws = [fw for fw in FW if any(k[0] == fw for k in runs)]
arms = [a for a in ARMS if any((fw, a) in runs for fw in fws)]
seeds_all = sorted({s for g in runs.values() for s in g})


def vals(group, key):
    v = np.array([m[key] for m in group.values()], dtype=float)
    return v[np.isfinite(v)]


def span(group, key, nd):
    v = vals(group, key)
    if len(v) == 0:
        return "-"
    if key.endswith("_ep") and len(group) == 1:
        nd = 0
    if len(group) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} [{v.min():.{nd}f}~{v.max():.{nd}f}]"


def mean_of(group, key):
    v = vals(group, key)
    return float(v.mean()) if len(v) else np.nan


def notes(arm, group):
    out = []
    if arm == BASE_ARM:
        out.append("기준선")
    n_div = sum(m["diverged"] for m in group.values())
    if n_div:
        out.append(f"발산 {n_div}/{len(group)}")
    n_cap = sum(m["capped"] for m in group.values())
    if n_cap == len(group):
        out.append("상한까지 안 멈춤")
    elif n_cap:
        out.append(f"상한까지 안 멈춤 {n_cap}/{len(group)}")
    return " · ".join(out)


# ---- 결과표 -----------------------------------------------------------------
COLS = [("train_acc", "train acc %", 1), ("val_acc", "val acc %", 1), ("gap", "train − val %p", 1),
        ("val_loss", "val loss 최저", 2), ("best_ep", "최저 에폭", 1), ("stop_ep", "멈춘 에폭", 1),
        ("test_acc", "test acc %", 1)]
for fw in fws:
    seeds_fw = sorted({s for a in arms for s in runs.get((fw, a), {})})
    bar(f"[결과표] {FW[fw]} — Dropout {BASE_DROPOUT}, Adam 1e-3, 얼리스탑(상한 {EPOCH_CAP}, patience {PATIENCE}), "
        f"시드 {', '.join(map(str, seeds_fw))}" + ("" if len(seeds_fw) == 1 else ", 평균 [최소~최대]"))
    print("   값은 얼리스탑이 돌려준 모델(val loss 최저 에폭)의 것이다. train acc는 증강 없이 다시 잰 값이다")
    head = ["증강", "시드"] + [name for _, name, _ in COLS] + ["비고"]
    body = []
    for a in arms:
        g = runs.get((fw, a))
        if not g:
            body.append([a, "결과 없음"] + [""] * (len(head) - 2))
            continue
        body.append([a, ",".join(map(str, sorted(g)))]
                    + [span(g, key, nd) for key, _, nd in COLS] + [notes(a, g)])
    ws = [max(width(r[i]) for r in [head] + body) + 2 for i in range(len(head) - 1)]
    for r in [head] + body:
        print("".join(pad(c, w) for c, w in zip(r[:-1], ws)) + r[-1])


# ---- 기준선(none) 재현 확인 -------------------------------------------------
def prev_best(fw, seed):
    """4단계 30에폭 곡선에서 val loss 최저 에폭, 그 값, 그 에폭의 val acc."""
    with open(PREV_RESULTS, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if (r["framework"], float(r["dropout"]), int(r["seed"])) == (fw, BASE_DROPOUT, seed):
                vl, va = json.loads(r["val_loss"]), json.loads(r["val_acc"])
                i = int(np.argmin(vl))
                return i + 1, vl[i], va[i] * 100
    return None


bar("[재현 확인] none 줄은 4단계 기준선과 같은 설정이다 — 최저 에폭이 같고 값이 재실행 흔들림(val 1~2%p) 안에서 맞아야 한다")
if not os.path.exists(PREV_RESULTS):
    print(f"   4단계 결과 파일({PREV_RESULTS})이 없어 비교하지 않는다 — 4단계 [곡선 전체] 줄과 직접 비교할 것")
else:
    for fw in fws:
        for s in sorted(runs.get((fw, BASE_ARM), {})):
            pb = prev_best(fw, s)
            if pb is None:
                print(f"   {FW[fw]:8s} 시드 {s}: 4단계에 Dropout {BASE_DROPOUT} 시드 {s} 실행이 없다")
                continue
            m = runs[(fw, BASE_ARM)][s]
            print(f"   {pad(FW[fw], 9)}시드 {s}  최저 에폭 {m['best_ep']} (4단계 {pb[0]}) · "
                  f"val loss 최저 {m['val_loss']:.2f} (4단계 {pb[1]:.2f}, {m['val_loss'] - pb[1]:+.2f}) · "
                  f"그 에폭의 val acc {m['val_acc']:.1f} (4단계 {pb[2]:.1f}, {m['val_acc'] - pb[2]:+.1f}%p)")


# ---- 같은 시드끼리 비교 -----------------------------------------------------
def paired(fw, a, b, key, higher_better):
    """같은 시드끼리 뺀 값 a-b로 줄 a와 b를 비교한다. higher_better=False면 값이 낮은 쪽이 낫다."""
    ga, gb = runs.get((fw, a), {}), runs.get((fw, b), {})
    seeds = [s for s in sorted(set(ga) & set(gb))
             if np.isfinite(ga[s][key]) and np.isfinite(gb[s][key])]
    out = dict(diffs=[], mean=np.nan, limit=np.nan, winner=None, enough=len(seeds) >= MIN_SEEDS,
               text="비교할 공통 시드가 없다")
    if not seeds:
        return out
    va = np.array([ga[s][key] for s in seeds])
    vb = np.array([gb[s][key] for s in seeds])
    d = va - vb
    out.update(diffs=[float(x) for x in d], mean=float(d.mean()))
    if not out["enough"]:
        out["text"] = f"판정 안 함 (공통 시드 {len(seeds)}개, {MIN_SEEDS}개 필요)"
        return out
    gain = d if higher_better else -d              # 양수 = a가 낫다
    limit = float(max(np.ptp(va), np.ptp(vb)))      # 시드 범위: 두 조건 중 큰 쪽
    out["limit"] = limit
    consistent = bool((gain > 0).all() or (gain < 0).all())
    if consistent and abs(float(gain.mean())) > limit:
        winner = "a" if gain.mean() > 0 else "b"
        out["winner"] = winner
        out["text"] = f"{a if winner == 'a' else b} 우세 (뚜렷함)"
    elif not consistent:
        out["text"] = (f"구분 안 됨 — 시드마다 방향이 다름 "
                       f"({a} 우세 {int((gain > 0).sum())}/{len(seeds)})")
    else:
        out["text"] = "구분 안 됨 — 방향은 같지만 평균 차이가 시드 범위 안"
    return out


def show_cmp(fw, a, b, key, higher_better, title, nd, unit, judge=True):
    c = paired(fw, a, b, key, higher_better)
    head = pad(FW[fw], 9) + pad(a, 10) + pad(title, 18)
    if not c["diffs"]:
        print(head + c["text"])
        return c
    per = " ".join(f"{x:+.{nd}f}" for x in c["diffs"])
    lim = f"  시드 범위 {c['limit']:.{nd}f}" if c["enough"] else ""
    tail = f"  -> {c['text']}" if judge else "  (보고용)"
    print(head + f"{c['mean']:+.{nd}f}{unit}  (시드별 {per}){lim}" + tail)
    return c


bar(f"[기준선 대비] 같은 시드끼리 뺀 값 (증강 줄 − {BASE_ARM}), val loss는 낮은 쪽이 낫다\n"
    "   뚜렷함 = 시드 전부 같은 방향이고 평균 차이가 시드 범위(두 조건의 최대−최소 중 큰 쪽)보다 클 것\n"
    "   시드가 3개 미만이면 판정하지 않고 차이만 보여 준다 (훑어보기)")
cells = {}
for a in arms:
    if a == BASE_ARM:
        continue
    cells[a] = []
    for fw in fws:
        cells[a].append(show_cmp(fw, a, BASE_ARM, "val_acc", True, "val acc", 1, "%p"))
        cells[a].append(show_cmp(fw, a, BASE_ARM, "val_loss", False, "val loss 최저", 3, ""))
        show_cmp(fw, a, BASE_ARM, "gap", False, "train − val 격차", 1, "%p", judge=False)
        show_cmp(fw, a, BASE_ARM, "test_acc", True, "test acc", 1, "%p", judge=False)
        show_cmp(fw, a, BASE_ARM, "best_ep", True, "최저 에폭", 1, "에폭", judge=False)
    print()

if cells:
    print(f"-> 판정 (4칸 = val acc · val loss 최저 × 두 프레임워크, {BASE_ARM} 대비)")
movers = []
all_hold = True
for a, cs in cells.items():
    n_new = sum(c["winner"] == "a" for c in cs)
    n_base = sum(c["winner"] == "b" for c in cs)
    n_skip = sum(not c["enough"] for c in cs)
    n_tie = len(cs) - n_new - n_base - n_skip
    if len(fws) < 2 or n_skip:
        label = "판정 보류"
    else:
        all_hold = False
        if n_new == len(cs):
            label = "기준선보다 뚜렷하게 낫다 -> 기준선에 넣을 후보"
            movers.append(a)
        elif n_base == len(cs):
            label = "기준선이 뚜렷하게 낫다"
        else:
            label = "구분 안 됨"
    extra = f" · 판정 안 함 {n_skip}칸" if n_skip else ""
    print(f"   {a}: {label}  ({a} 우세 {n_new}칸 · 기준선 우세 {n_base}칸 · 구분 안 됨 {n_tie}칸{extra})")
print()
if not cells:
    print(f"-> 결론: 기준선({BASE_ARM})뿐이라 비교할 증강이 없다 — 위 표의 {BASE_ARM} 행이 [재현 확인]과 맞는지부터 본다")
elif all_hold:
    print("-> 결론: 판정 보류 — 두 프레임워크 모두 시드 3개씩이 필요하다. 지금은 훑어보기이므로 큰 차이만 읽는다")
elif movers:
    def score(a):
        return np.mean([mean_of(runs[(fw, a)], "val_acc") for fw in fws if (fw, a) in runs])
    pick = max(movers, key=score)
    print(f"-> 결론: 증강 {pick}을 기준선에 넣는다 (4칸 전부에서 {BASE_ARM}보다 뚜렷하게 앞섰다"
          + (f", 후보 {', '.join(movers)} 중 val acc 평균이 가장 높다)" if len(movers) > 1 else ")"))
else:
    print("-> 결론: 증강 없음 유지 — 어느 증강도 4칸 전부에서 기준선보다 뚜렷하게 앞서지 않았다")

# ---- 그래프 -----------------------------------------------------------------
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
ARM_COLOR = {"none": "#b0489d", "flip": "#2a78d6", "rotation": "#d55e00", "zoom": "#009e73"}
ARM_MARK = {"none": "o", "flip": "s", "rotation": "^", "zoom": "D"}
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False


def style(ax, grid_axis="y"):
    ax.set_facecolor(SURFACE)
    ax.grid(True, axis=grid_axis, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(AXIS)
    ax.tick_params(colors=MUTED, labelsize=9)


# 그래프 1 — 에폭별 곡선 (줄마다 색 하나, 점선 = 기준선, 표시 = 얼리스탑이 돌려준 에폭)
fig, axes = plt.subplots(len(fws), 2, figsize=(11, 3.8 * len(fws) + 1.0), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (name, key) in enumerate((("val loss", "val_loss"), ("val acc (%)", "val_acc"))):
        ax = axes[i][j]
        style(ax, "y")
        for a in arms:
            group = curves.get((fw, a))
            if not group:
                continue
            color, ls = ARM_COLOR[a], ("--" if a == BASE_ARM else "-")
            many = len(group) > 1
            for c in group.values():
                y = c[key]
                ax.plot(np.arange(1, len(y) + 1), y, color=color, ls=ls, lw=0.9 if many else 2.4,
                        alpha=0.35 if many else 1.0, zorder=2)
                if c["best"] > 0:
                    ax.plot([c["best"]], [y[c["best"] - 1]], ls="none", marker=ARM_MARK[a], ms=7,
                            mfc=color, mec=SURFACE, mew=1.0, zorder=5)
            n = min(len(c[key]) for c in group.values())
            if many and n > 0:
                ax.plot(np.arange(1, n + 1), np.mean([c[key][:n] for c in group.values()], axis=0),
                        color=color, ls=ls, lw=2.4, zorder=3)
        ax.set_xlabel("epoch", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=ARM_COLOR[a], ls="--" if a == BASE_ARM else "-", lw=2.4,
                  marker=ARM_MARK[a], ms=7, mec=SURFACE, label=a + (" (base)" if a == BASE_ARM else ""))
           for a in arms]
fig.text(0.005, 0.99, f"Learning curves by augmentation, early stopping (cap {EPOCH_CAP}, patience {PATIENCE})  -  "
         "marker = epoch of the returned model", fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("aug_curves.png", facecolor=SURFACE)
plt.show()
print("[그래프] aug_curves.png")

# 그래프 2 — 줄별 결과 (점 하나 = 시드 하나)
PANELS = [("val_acc", "val acc (%) of the returned model", 1.0),
          ("val_loss", "best val loss", 0.03),
          ("gap", "train (clean) - val acc gap (%p)", 1.0)]
fig, axes = plt.subplots(len(fws), 3, figsize=(13, 3.6 * len(fws) + 0.9), dpi=150,
                         squeeze=False, facecolor=SURFACE)
for i, fw in enumerate(fws):
    for j, (key, name, min_pad) in enumerate(PANELS):
        ax = axes[i][j]
        style(ax)
        seen = []
        for x, a in enumerate(arms):
            pts = [(s, m[key]) for s, m in sorted(runs.get((fw, a), {}).items()) if np.isfinite(m[key])]
            if not pts:
                continue
            color = ARM_COLOR[a]
            ys = np.array([p[1] for p in pts])
            seen += list(ys)
            ax.vlines(x, ys.min(), ys.max(), color=color, lw=1.6, alpha=0.45, zorder=2)
            ax.plot([x - 0.3, x + 0.3], [ys.mean()] * 2, color=color, lw=2.4,
                    solid_capstyle="butt", zorder=3)
            ax.plot([x + 0.2 * (s - 1) for s, _ in pts], ys, ls="none", marker="o", ms=6.5,
                    mfc=color, mec=SURFACE, mew=1.1, zorder=4)
        if seen:
            lo, hi = min(seen), max(seen)
            margin = max(min_pad, (hi - lo) * 0.2)
            ax.set_ylim(lo - margin, hi + margin)
        ax.set_xlim(-0.6, len(arms) - 0.4)
        ax.set_xticks(range(len(arms)))
        ax.set_xticklabels([a + (" (base)" if a == BASE_ARM else "") for a in arms])
        ax.set_xlabel("augmentation", fontsize=9, color=INK2)
        ax.set_title(f"{FW[fw]} · {name}", fontsize=10.5, color=INK, loc="left")
handles = [Line2D([0], [0], color=INK2, lw=2.4, label="mean"),
           Line2D([0], [0], marker="o", ls="none", ms=6.5, mfc=INK2, mec=SURFACE, mew=1.1,
                  label="each seed (left to right: 0, 1, 2)")]
fig.text(0.005, 0.99, f"Augmentation (one type at a time), dropout {BASE_DROPOUT}, Adam 1e-3, He init, early stopping",
         fontsize=12, color=INK, va="top")
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 0.955),
           ncol=len(handles), frameon=False, fontsize=9, labelcolor=INK2)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("aug_seeds.png", facecolor=SURFACE)
plt.show()
print("[그래프] aug_seeds.png")